# **Install dependencies**

In [ ]:
!pip install requests feedparser pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 1.7 MB/s eta 0:00:00


# **Imports + a helper to inspect any source's fields**

In [ ]:
import requests
import feedparser
import pandas as pd
import json
from datetime import datetime

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; TrendRadarBot/0.1; capstone project)"}

def inspect_source(source_name, records, raw_sample=None):
    """
    Given a list of dict records from a source, print:
    - how many records came back
    - the field names available
    - one raw sample record (pretty-printed)
    """
    print(f"\n{'='*70}")
    print(f"SOURCE: {source_name}")
    print(f"{'='*70}")
    if not records:
        print("⚠️  No records returned — check the request/feed above for errors.")
        return
    print(f"Records fetched: {len(records)}")
    print(f"Fields found: {list(records[0].keys())}")
    print("\n--- Sample record ---")
    sample = raw_sample if raw_sample is not None else records[0]
    print(json.dumps(sample, indent=2, default=str))  # trimmed for readability

# **dev.to — official public API, no key needed**

In [ ]:
def get_most_popular_devto_tag():
    """Fetches the most popular tag from dev.to API."""
    url = "https://dev.to/api/tags"
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    tags_data = resp.json()
    if tags_data:
        # The API returns tags sorted by articles_count by default,
        # so the first one is the most popular.
        return tags_data[0].get("name")
    return ""

def test_devto(tag="", per_page=1000): # Changed default per_page to a high number to test API limit
    url = f"https://dev.to/api/articles?tag={tag}&per_page={per_page}"
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    records = [{
        "article_id": item.get("id"),
        "title": item.get("title"),
        "author": item.get("user", {}).get("name"),
        "published_date": item.get("published_at"),
        "url": item.get("url"),
        "tags": item.get("tag_list"),
        "reactions_count": item.get("public_reactions_count"),
        "comments_count": item.get("comments_count"),
        "summary": item.get("description"),
    } for item in data]
    return records, data[0] if data else None

# Get a trending tag dynamically
trending_tag = get_most_popular_devto_tag()
print(f"Using trending dev.to tag: {trending_tag}")

# Call test_devto with the trending tag and the higher per_page limit
devto_records, devto_raw = test_devto(tag=trending_tag, per_page=1000)
inspect_source(f"dev.to (tag: {trending_tag}, per_page=1000)", devto_records, devto_raw)


Using trending dev.to tag: webdev

SOURCE: dev.to (tag: webdev, per_page=1000)
Records fetched: 1000
Fields found: ['article_id', 'title', 'author', 'published_date', 'url', 'tags', 'reactions_count', 'comments_count', 'summary']

--- Sample record ---
{
  "type_of": "article",
  "id": 4603458,
  "title": "Most 'AI Agents' Are Just If-Statements in a Trench Coat",
  "description": "I built an agent last year, and I was proud of it.  It had a planner. It had tools. It had a...",
  "readable_publish_date": "Sep 8",
  "slug": "most-ai-agents-are-just-if-statements-in-a-trench-coat-3960",
  "path": "/james_anderson_h/most-ai-agents-are-just-if-statements-in-a-trench-coat-3960",
  "url": "https://dev.to/james_anderson_h/most-ai-agents-are-just-if-statements-in-a-trench-coat-3960",
  "comments_count": 38,
  "public_reactions_count": 44,
  "collection_id": null,
  "published_timestamp": "2026-09-08T06:12:03Z",
  "language": "en",
  "subforem_id": 1,
  "ai_disclosure_level": "not_disclosed",
 

# **Medium — official RSS feed (per tag or per author), no key needed**

In [ ]:
def test_medium_rss(feed_url="https://medium.com/feed/tag/data-engineering"):
    feed = feedparser.parse(feed_url)
    records = [{
        "title": entry.get("title"),
        "author": entry.get("author"),
        "published_date": entry.get("published"),
        "url": entry.get("link"),
        "tags": [t["term"] for t in entry.get("tags", [])] if entry.get("tags") else None,
        "summary": entry.get("summary", "")[:200],  # trimmed, HTML included
    } for entry in feed.entries]
    return records

# Try a tag feed. You can also try an author feed, e.g.:
# "https://medium.com/feed/@username"
medium_records = test_medium_rss()
inspect_source("Medium (tag: data-engineering)", medium_records)


SOURCE: Medium (tag: data-engineering)
Records fetched: 10
Fields found: ['title', 'author', 'published_date', 'url', 'tags', 'summary']

--- Sample record ---
{
  "title": "Stop Rebuilding Your Whole dbt Project: Run Only What Changed",
  "author": "Rashmi Mohapatra",
  "published_date": "Wed, 09 Sep 2026 14:16:06 GMT",
  "url": "https://medium.com/@rashmeemayee.mohapatra/stop-rebuilding-your-whole-dbt-project-run-only-what-changed-cbfc1dcd092a?source=rss------data_engineering-5",
  "tags": [
    "data-engineering",
    "dbt",
    "design-patterns",
    "testing",
    "data"
  ],
  "summary": "<div class=\"medium-feed-item\"><p class=\"medium-feed-image\"><a href=\"https://medium.com/@rashmeemayee.mohapatra/stop-rebuilding-your-whole-dbt-project-run-only-what-changed-cbfc1dcd092a?source=rss-----"
}


# **Towards Data Science — Medium publication RSS feed**

In [ ]:
def test_tds_rss():
    return test_medium_rss(feed_url="https://towardsdatascience.com/feed")

tds_records = test_tds_rss()
inspect_source("Towards Data Science (Medium publication)", tds_records)


SOURCE: Towards Data Science (Medium publication)
Records fetched: 20
Fields found: ['title', 'author', 'published_date', 'url', 'tags', 'summary']

--- Sample record ---
{
  "title": "The Symmetry That Breaks Neural Network Averaging",
  "author": "Ananya Bhattacharyya",
  "published_date": "Wed, 09 Sep 2026 14:00:02 GMT",
  "url": "https://towardsdatascience.com/the-symmetry-that-breaks-neural-network-averaging/",
  "tags": [
    "Deep Learning",
    "Neural Networks",
    "Optimisation",
    "Neural Network Training",
    "Machine Learning",
    "Editor's Picks"
  ],
  "summary": "<p>Understanding permutation symmetry in deep learning, and what it means for weight averaging and model merging</p>\n<p>The post <a href=\"https://towardsdatascience.com/the-symmetry-that-breaks-neural"
}


# **The Guardian — official Open Platform API (needs a free API key)**

In [ ]:
# Register free at: https://open-platform.theguardian.com/access/
# The shared "test" key works for light testing but is rate-limited/shared —
# get your own key before building the real pipeline.
GUARDIAN_API_KEY = "test"  # <-- replace with your own key

def test_guardian(query="data engineering", api_key=GUARDIAN_API_KEY, page_size=50): # Increased page_size to 50 to test API limit
    url = "https://content.guardianapis.com/search"
    params = {
        "q": query,
        "api-key": api_key,
        "page-size": page_size,
        "show-fields": "trailText,byline",
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()["response"]["results"]
    records = [{
        "article_id": item.get("id"),
        "title": item.get("webTitle"),
        "section": item.get("sectionName"),
        "published_date": item.get("webPublicationDate"),
        "url": item.get("webUrl"),
        "author": item.get("fields", {}).get("byline"),
        "summary": item.get("fields", {}).get("trailText"),
    } for item in data]
    return records, data[0] if data else None

guardian_records, guardian_raw = test_guardian()
inspect_source("The Guardian API", guardian_records, guardian_raw)


_IncompleteInputError: incomplete input (1439024001.py, line 10)

# **BBC — official RSS feed (do NOT scrape bbc.co.uk HTML directly)**

In [ ]:
def test_bbc_rss(feed_url="https://feeds.bbci.co.uk/news/technology/rss.xml"):
    feed = feedparser.parse(feed_url)
    records = [{
        "title": entry.get("title"),
        "published_date": entry.get("published"),
        "url": entry.get("link"),
        "summary": entry.get("summary"),
    } for entry in feed.entries]
    return records

bbc_records = test_bbc_rss()
inspect_source("BBC (Technology RSS)", bbc_records)


SOURCE: BBC (Technology RSS)
Records fetched: 20
Fields found: ['title', 'published_date', 'url', 'summary']

--- Sample record ---
{
  "title": "Amazon rigged billions in ad pricing, lawsuit from states and US watchdog alleges",
  "published_date": "Mon, 31 Aug 2026 21:28:49 GMT",
  "url": "https://www.bbc.co.uk/news/articles/cvgy91nvy27o?at_medium=RSS&at_campaign=rss",
  "summary": "Amazon responded to the lawsuit, arguing the US Federal Trade Commission \"misunderstands\" their ad market."
}


# **Hacker News — Algolia HN Search API (easiest way to search by keyword)**

In [ ]:
def test_hackernews(query="data engineering", hits_per_page=5):
    url = "https://hn.algolia.com/api/v1/search"
    params = {"query": query, "tags": "story", "hitsPerPage": hits_per_page}
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()["hits"]
    records = [{
        "article_id": item.get("objectID"),
        "title": item.get("title"),
        "author": item.get("author"),
        "published_date": item.get("created_at"),
        "url": item.get("url"),
        "points": item.get("points"),
        "num_comments": item.get("num_comments"),
    } for item in data]
    return records, data[0] if data else None

hn_records, hn_raw = test_hackernews()
inspect_source("Hacker News (Algolia API)", hn_records, hn_raw)


SOURCE: Hacker News (Algolia API)
Records fetched: 5
Fields found: ['article_id', 'title', 'author', 'published_date', 'url', 'points', 'num_comments']

--- Sample record ---
{
  "_highlightResult": {
    "author": {
      "matchLevel": "none",
      "matchedWords": [],
      "value": "charlysl"
    },
    "title": {
      "fullyHighlighted": false,
      "matchLevel": "full",
      "matchedWords": [
        "data",
        "engineering"
      ],
      "value": "<em>Data</em> <em>Engineering</em> Cookbook"
    },
    "url": {
      "matchLevel": "none",
      "matchedWords": [],
      "value": "https://github.com/andkret/Cookbook"
    }
  },
  "_tags": [
    "story",
    "author_charlysl",
    "story_20554194"
  ],
  "author": "charlysl",
  "children": [
    20554711,
    20554714,
    20554777,
    20554885,
    20554949,
    20554993,
    20555334,
    20555346,
    20555442,
    20555633,
    20556312,
    20556615,
    20557542,
    20557832,
    20558452,
    20559378,
    206244

# **Stack Overflow — two options: Blog RSS, and Stack Exchange Q&A API**

In [ ]:
def test_so_blog_rss():
    feed = feedparser.parse("https://stackoverflow.blog/feed/")
    records = [{
        "title": entry.get("title"),
        "author": entry.get("author"),
        "published_date": entry.get("published"),
        "url": entry.get("link"),
        "summary": entry.get("summary", "")[:200],
    } for entry in feed.entries]
    return records

def test_stackexchange_api(tag="data-engineering", pagesize=5):
    url = "https://api.stackexchange.com/2.3/questions"
    params = {
        "order": "desc", "sort": "activity", "tagged": tag,
        "site": "stackoverflow", "pagesize": pagesize
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()["items"]
    records = [{
        "question_id": item.get("question_id"),
        "title": item.get("title"),
        "author": item.get("owner", {}).get("display_name"),
        "published_date": item.get("creation_date"),  # unix timestamp
        "url": item.get("link"),
        "tags": item.get("tags"),
        "answer_count": item.get("answer_count"),
        "score": item.get("score"),
    } for item in data]
    return records, data[0] if data else None

so_blog_records = test_so_blog_rss()
inspect_source("Stack Overflow Blog (RSS)", so_blog_records)

so_qa_records, so_qa_raw = test_stackexchange_api()
inspect_source("Stack Exchange Q&A API (tag: data-engineering)", so_qa_records, so_qa_raw)


SOURCE: Stack Overflow Blog (RSS)
Records fetched: 40
Fields found: ['title', 'author', 'published_date', 'url', 'summary']

--- Sample record ---
{
  "title": "When you keep AI Lean, you keep AI correct",
  "author": "Phoebe Sajor",
  "published_date": "Fri, 28 Aug 2026 07:40:00 GMT",
  "url": "https://stackoverflow.blog/2026/08/28/when-you-keep-ai-lean-you-keep-ai-correct/",
  "summary": "Ryan chats with Leo de Moura, Senior Principal Applied Scientist at AWS and the creator of the Lean language, about proving correctness in AI agents with the Lean language, how automated reasoning com"
}

SOURCE: Stack Exchange Q&A API (tag: data-engineering)
Records fetched: 5
Fields found: ['question_id', 'title', 'author', 'published_date', 'url', 'tags', 'answer_count', 'score']

--- Sample record ---
{
  "tags": [
    "advice",
    "best-practices",
    "etl",
    "aws-glue",
    "data-engineering"
  ],
  "owner": {
    "account_id": 45160670,
    "reputation": 1,
    "user_id": 32126869,
    

# **KDnuggets — official RSS feed**

In [ ]:
def test_kdnuggets_rss():
    feed = feedparser.parse("https://www.kdnuggets.com/feed")
    records = [{
        "title": entry.get("title"),
        "author": entry.get("author"),
        "published_date": entry.get("published"),
        "url": entry.get("link"),
        "tags": [t["term"] for t in entry.get("tags", [])] if entry.get("tags") else None,
        "summary": entry.get("summary", "")[:200],
    } for entry in feed.entries]
    return records

kdnuggets_records = test_kdnuggets_rss()
inspect_source("KDnuggets (RSS)", kdnuggets_records)


SOURCE: KDnuggets (RSS)
Records fetched: 10
Fields found: ['title', 'author', 'published_date', 'url', 'tags', 'summary']

--- Sample record ---
{
  "title": "Speed Up LLM Inference with DSpark Speculative Decoding",
  "author": "Abid Ali Awan",
  "published_date": "Mon, 31 Aug 2026 14:00:00 +0000",
  "url": "https://www.kdnuggets.com/speed-up-llm-inference-with-dspark-speculative-decoding",
  "tags": null,
  "summary": "Learn how DSpark speculative decoding can improve local LLM generation speed using the same GPU, with Qwen3-8B, llama.cpp, and CUDA."
}


# **Build a summary table for Source_Fields sheet**

In [ ]:
# This collects field names per source into one comparison DataFrame.
source_field_map = {
    "dev.to": list(devto_records[0].keys()) if devto_records else [],
    "Medium (tag feed)": list(medium_records[0].keys()) if medium_records else [],
    "Towards Data Science": list(tds_records[0].keys()) if tds_records else [],
    "The Guardian API": list(guardian_records[0].keys()) if guardian_records else [],
    "BBC (RSS)": list(bbc_records[0].keys()) if bbc_records else [],
    "Hacker News (Algolia)": list(hn_records[0].keys()) if hn_records else [],
    "Stack Overflow Blog (RSS)": list(so_blog_records[0].keys()) if so_blog_records else [],
    "Stack Exchange API": list(so_qa_records[0].keys()) if so_qa_records else [],
    "KDnuggets (RSS)": list(kdnuggets_records[0].keys()) if kdnuggets_records else [],
}

summary_df = pd.DataFrame([
    {"source": src, "num_fields": len(fields), "fields": ", ".join(fields)}
    for src, fields in source_field_map.items()
])
print("\n\n===== SOURCE FIELD SUMMARY (for 03_Source_Fields sheet) =====")
print(summary_df.to_string(index=False))

# Optional: save it as a CSV you can paste straight into your workbook
summary_df.to_csv("source_field_summary.csv", index=False)
print("\nSaved: source_field_summary.csv")



===== SOURCE FIELD SUMMARY (for 03_Source_Fields sheet) =====
                   source  num_fields                                                                                         fields
                   dev.to           9 article_id, title, author, published_date, url, tags, reactions_count, comments_count, summary
        Medium (tag feed)           6                                              title, author, published_date, url, tags, summary
     Towards Data Science           6                                              title, author, published_date, url, tags, summary
         The Guardian API           7                               article_id, title, section, published_date, url, author, summary
                BBC (RSS)           4                                                            title, published_date, url, summary
    Hacker News (Algolia)           7                           article_id, title, author, published_date, url, points, num_comments
Stack

In [1]:
# Quick notes to carry into Source_Fields sheet
print("""
NOTES TO RECORD PER SOURCE:
- Type of each field (string / datetime / int / list) — check the sample JSON above.
- Personal data check: these sources only expose PUBLIC byline/author names on
  public content (not private user data) — generally not a personal-data concern,
  but note it in the sheet since author name is still a name field.
- Medium RSS feeds only return the ~10 most recent items per feed/tag — factor
  that into how often you schedule polling.
- The Guardian 'test' API key is shared/rate-limited — get your own free key
  before building the real pipeline: https://open-platform.theguardian.com/access/
- BBC/KDnuggets/Medium/TDS are all RSS — lighter weight, no auth, but summaries
  are often truncated/HTML — full article text is NOT included, only excerpts.
""")


NOTES TO RECORD PER SOURCE:
- Type of each field (string / datetime / int / list) — check the sample JSON above.
- Personal data check: these sources only expose PUBLIC byline/author names on
  public content (not private user data) — generally not a personal-data concern,
  but note it in the sheet since author name is still a name field.
- Medium RSS feeds only return the ~10 most recent items per feed/tag — factor
  that into how often you schedule polling.
- The Guardian 'test' API key is shared/rate-limited — get your own free key
  before building the real pipeline: https://open-platform.theguardian.com/access/
- BBC/KDnuggets/Medium/TDS are all RSS — lighter weight, no auth, but summaries
  are often truncated/HTML — full article text is NOT included, only excerpts.



# **Load API response into JSON**

In [ ]:
import json

output_filename = "devto_1k_records_full.json"

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(devto_records, f, ensure_ascii=False, indent=2)

print(f"All dev.to API records (1000 records) saved to {output_filename}")

All dev.to API records (1000 records) saved to devto_1k_records_full.json
